# Proyecto de regresión logística

En este notebook voy a hacer el proyecto de regresión logística paso a paso.  
La idea es entender mejor los datos, hacer un pequeño EDA, preparar el dataset correctamente y luego entrenar un modelo sencillo para predecir la variable objetivo.


## 1. Importar librerías

Primero importo todas las librerías que voy a usar en el proyecto.  
Las dejo juntas desde el principio para tener el notebook más ordenado.


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)
from pickle import dump

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)


## 2. Cargar los datos

Aquí cargo el dataset desde la URL del ejercicio.  
Así trabajo directamente con los datos sin necesidad de descargarlos manualmente.


In [ ]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/logistic-regression-project-tutorial/main/bank-marketing-campaign-data.csv"
total_data = pd.read_csv(url, sep=";")
total_data.head()


## 3. Revisión rápida

Antes de empezar con el modelo, reviso el tamaño del dataset, los duplicados, los nulos y los tipos de datos para entender un poco mejor qué tengo delante.


In [ ]:
total_data.shape


In [ ]:
total_data.duplicated().sum()


In [ ]:
total_data.isnull().sum()


In [ ]:
total_data.dtypes


## 4. Limpieza inicial

En este caso no hay nulos, pero sí hay algunos registros duplicados, así que los elimino antes de seguir.


In [ ]:
total_data = total_data.drop_duplicates().reset_index(drop=True)
total_data.shape


## 5. EDA de la variable objetivo

Primero miro cómo está repartida la variable objetivo.  
Esto es importante porque si las clases están desbalanceadas luego conviene tenerlo en cuenta en el split y en el modelo.


In [ ]:
total_data["y"].value_counts()


In [ ]:
(total_data["y"].value_counts(normalize=True) * 100).round(2)


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=total_data, x="y")
plt.title("Distribución de la variable objetivo")
plt.xlabel("y")
plt.ylabel("Frecuencia")
plt.show()


## 6. EDA de variables numéricas

Aquí reviso las variables numéricas con histogramas para ver su distribución y también hago boxplots para detectar posibles valores extremos.


In [ ]:
numeric_cols = total_data.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols


In [ ]:
total_data[numeric_cols].hist(figsize=(16, 12), bins=30)
plt.suptitle("Distribución de variables numéricas", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(16, 8))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(3, 4, i)
    sns.boxplot(y=total_data[col])
    plt.title(col)
plt.tight_layout()
plt.show()


## 7. EDA de variables categóricas

También reviso algunas de las variables categóricas más importantes para ver cómo están distribuidas.


In [ ]:
categorical_cols = total_data.select_dtypes(include=["object"]).columns.tolist()
categorical_cols


In [ ]:
eda_cats = ["job", "marital", "education", "contact", "month", "poutcome"]

plt.figure(figsize=(18, 18))
for i, col in enumerate(eda_cats, 1):
    plt.subplot(3, 2, i)
    order = total_data[col].value_counts().index
    sns.countplot(data=total_data, y=col, order=order)
    plt.title(f"Distribución de {col}")
plt.tight_layout()
plt.show()


## 8. Heatmap de relaciones

Para ver relaciones entre variables numéricas, hago un heatmap de correlación.  
Aquí convierto la variable objetivo a 0 y 1 solo para poder incluirla en esta visualización.


In [ ]:
eda_data = total_data.copy()
eda_data["y_num"] = eda_data["y"].map({"no": 0, "yes": 1})

corr_cols = numeric_cols + ["y_num"]
corr_matrix = eda_data[corr_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Heatmap de correlación")
plt.show()


## 9. Decisiones antes del modelado

Después de revisar los datos, tomo algunas decisiones:

- Elimino duplicados.
- La variable objetivo está desbalanceada, así que usaré `stratify` en el split.
- También probaré `class_weight='balanced'` en la regresión logística.
- No voy a usar `factorize` para las variables categóricas, porque no es la mejor opción para este modelo.
- En su lugar usaré `OneHotEncoder` dentro de un pipeline.
- Además elimino la variable `duration`, porque puede producir data leakage: esa información solo se conoce al final de la llamada y no sería válida en un escenario real de predicción.


## 10. Preparar X e y

Aquí separo las variables predictoras y la variable objetivo.  
También elimino `duration` por el problema de leakage comentado antes.


In [ ]:
model_data = total_data.copy()
model_data = model_data.drop(columns=["duration"])

X = model_data.drop(columns=["y"])
y = model_data["y"].map({"no": 0, "yes": 1})

X.head()


## 11. Train y test

Ahora hago la separación entre train y test.  
En este caso uso `stratify` para mantener la proporción de clases, y dejo un 20% para test.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train.shape, X_test.shape


In [ ]:
y_train.value_counts(normalize=True).round(4), y_test.value_counts(normalize=True).round(4)


## 12. Preprocesado

Como tengo variables numéricas y categóricas, preparo un pipeline para tratarlas de forma correcta.  
Las numéricas las imputo y escalo, y las categóricas las imputo y las transformo con One Hot Encoding.


In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 13. Selección de variables y modelo base del pipeline

Aquí monto ya el pipeline completo con el preprocesado, la selección de variables y la regresión logística.  
La selección la hago con `SelectKBest` y después probaré varios valores de `k`.


In [ ]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selectk", SelectKBest(score_func=chi2)),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])


## 14. Pequeña optimización

Pruebo varios valores de `k`, distintos valores de `C` y dos solvers diferentes para ver qué combinación funciona mejor.


In [ ]:
grid = GridSearchCV(
    pipeline,
    {
        "selectk__k": [3, 5, 7, 10],
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["lbfgs", "liblinear"]
    },
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)
grid.best_params_


In [ ]:
grid.best_score_


## 15. Evaluación del mejor modelo

Una vez encontrada la mejor combinación, evalúo el modelo en el conjunto de test.


In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_pred[:10]


In [ ]:
accuracy_score(y_test, y_pred)


In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_proba)


In [ ]:
confusion_matrix(y_test, y_pred)


In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred))
disp.plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()


## 16. Variables seleccionadas

Como estoy usando un pipeline con codificación y selección de variables, aquí saco los nombres de las columnas que finalmente se han quedado en el modelo.


In [ ]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
selected_mask = best_model.named_steps["selectk"].get_support()
selected_features = feature_names[selected_mask]

selected_features


## 17. Guardar datos limpios

Guardo una copia del train y del test por si luego quiero reutilizarlos por separado.


In [ ]:
os.makedirs("data/processed", exist_ok=True)

train_data = X_train.copy()
train_data["y"] = y_train.values

test_data = X_test.copy()
test_data["y"] = y_test.values

train_data.to_csv("data/processed/clean_train.csv", index=False)
test_data.to_csv("data/processed/clean_test.csv", index=False)


In [ ]:
pd.read_csv("data/processed/clean_train.csv").head()


## 18. Guardar el modelo

Por último guardo el pipeline completo con el mejor modelo encontrado.


In [ ]:
os.makedirs("models", exist_ok=True)
dump(best_model, open("models/logistic_regression_pipeline.sav", "wb"))


## Conclusión

En este proyecto he cargado el dataset, he hecho una revisión inicial, un pequeño EDA con visualizaciones, he limpiado los datos y he preparado correctamente las variables para entrenar una regresión logística.

También he tenido en cuenta el desbalanceo de la variable objetivo usando `stratify` en la separación de train y test y `class_weight='balanced'` en el modelo.  
Además, para evitar problemas de data leakage, he eliminado la variable `duration` y he hecho el preprocesado dentro de un pipeline.

Por último, he probado varias configuraciones con `GridSearchCV` para quedarme con la mejor combinación posible dentro de este enfoque.
